In [1]:
import pandas as pd

from immunodash.client import ChEMBLClient
from immunodash.targets import get_target
from immunodash.activities import get_activities
from immunodash.cleaning import (
    convert_numeric_columns,
    filter_standardized,
    filter_activity_types,
    remove_missing_measurements,
    clean_activities,
)

In [2]:
client = ChEMBLClient()

jak1 = get_target("JAK1")

activities = client.get_all(
    endpoint="activity",
    response_key="activities",
    params={
        "target_chembl_id": jak1["target_chembl_id"]
    },
    show_progress=True
)

raw_df = pd.DataFrame(activities)

[██████████████████████████████] 47,757/47,757(100.0%)


In [3]:
numeric_df = convert_numeric_columns(raw_df)

In [4]:
raw_df.dtypes

action_type                   object
activity_comment                 str
activity_id                    int64
activity_properties           object
assay_chembl_id                  str
assay_description                str
assay_type                       str
assay_variant_accession       object
assay_variant_mutation        object
bao_endpoint                     str
bao_format                       str
bao_label                        str
canonical_smiles                 str
data_validity_comment            str
data_validity_description        str
document_chembl_id               str
document_journal                 str
document_year                float64
ligand_efficiency             object
modality                      object
molecule_chembl_id               str
molecule_pref_name               str
parent_molecule_chembl_id        str
pchembl_value                    str
potential_duplicate            int64
qudt_units                       str
record_id                      int64
r

In [5]:
numeric_df.dtypes

action_type                   object
activity_comment                 str
activity_id                    int64
activity_properties           object
assay_chembl_id                  str
assay_description                str
assay_type                       str
assay_variant_accession       object
assay_variant_mutation        object
bao_endpoint                     str
bao_format                       str
bao_label                        str
canonical_smiles                 str
data_validity_comment            str
data_validity_description        str
document_chembl_id               str
document_journal                 str
document_year                  Int64
ligand_efficiency             object
modality                      object
molecule_chembl_id               str
molecule_pref_name               str
parent_molecule_chembl_id        str
pchembl_value                float64
potential_duplicate            int64
qudt_units                       str
record_id                      int64
r

In [6]:
standardized_df = filter_standardized(numeric_df)

In [7]:
standardized_df["standard_flag"].value_counts()

standard_flag
1    21315
Name: count, dtype: int64

In [8]:
potency_df = filter_activity_types(standardized_df)

In [9]:
potency_df["standard_type"].value_counts()

standard_type
IC50    14914
Ki       3890
Kd        563
EC50      427
Name: count, dtype: int64

In [10]:
complete_df = remove_missing_measurements(potency_df)

In [11]:
REQUIRED_COLUMNS = [
    "canonical_smiles",
    "standard_value",
    "standard_units",
]

complete_df[REQUIRED_COLUMNS].isna().sum()

canonical_smiles    0
standard_value      0
standard_units      0
dtype: int64

In [12]:
final_df = clean_activities(raw_df)

In [13]:
complete_df

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,NaN,129349,[],CHEMBL704110,Inhibition of Janus kinase 1,B,None,None,BAO_0000190,...,Homo sapiens,Tyrosine-protein kinase JAK1,9606,NaN,None,IC50,uM,UO_0000065,NaN,10.0000
1,None,NaN,972649,[],CHEMBL857179,Inhibition of Janus kinase 1,B,None,None,BAO_0000190,...,Homo sapiens,Tyrosine-protein kinase JAK1,9606,NaN,None,Log IC50,NaN,UO_0000065,NaN,-4.4000
2,None,NaN,978695,[],CHEMBL857179,Inhibition of Janus kinase 1,B,None,None,BAO_0000190,...,Homo sapiens,Tyrosine-protein kinase JAK1,9606,NaN,None,Log IC50,NaN,UO_0000065,NaN,-4.7000
3,None,NaN,1650687,[],CHEMBL860904,Average Binding Constant for JAK1 (Kin.Dom. 1)...,B,None,None,BAO_0000034,...,Homo sapiens,Tyrosine-protein kinase JAK1,9606,NaN,None,Kd,uM,UO_0000065,NaN,0.0092
4,None,NaN,1815348,[],CHEMBL910181,Inhibition of Jak1,B,None,None,BAO_0000190,...,Homo sapiens,Tyrosine-protein kinase JAK1,9606,NaN,None,IC50,nM,UO_0000065,NaN,1300.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19744,"{'action_type': 'INHIBITOR', 'description': 'N...",NaN,29170048,[],CHEMBL6109396,Binding affinity to JAK1 JH2 domain (unknown o...,B,None,None,BAO_0000034,...,Homo sapiens,Tyrosine-protein kinase JAK1,9606,NaN,None,Kd,nM,UO_0000065,NaN,3.7000
19745,None,NaN,29179764,[],CHEMBL6112055,Binding affinity to JAK1 (unknown origin) by A...,B,None,None,BAO_0000190,...,Homo sapiens,Tyrosine-protein kinase JAK1,9606,NaN,None,IC50,nM,UO_0000065,NaN,10000.0000
19746,None,NaN,29179770,[],CHEMBL6112059,Displacement of fluorophore-labeled probe from...,B,None,None,BAO_0000190,...,Homo sapiens,Tyrosine-protein kinase JAK1,9606,NaN,None,IC50,nM,UO_0000065,NaN,100.0000
19747,"{'action_type': 'INHIBITOR', 'description': 'N...",NaN,29179771,[],CHEMBL6112059,Displacement of fluorophore-labeled probe from...,B,None,None,BAO_0000190,...,Homo sapiens,Tyrosine-protein kinase JAK1,9606,NaN,None,IC50,nM,UO_0000065,NaN,1.8000


In [14]:
final_df.equals(complete_df)

True

In [15]:
sorted(final_df["standard_type"].unique())

['EC50', 'IC50', 'Kd', 'Ki']

In [16]:
final_df["molecule_chembl_id"].nunique()

12929

In [17]:
final_df["canonical_smiles"].nunique()

12929

In [18]:
final_df["molecule_chembl_id"].value_counts().head(20)

molecule_chembl_id
CHEMBL221959     83
CHEMBL1789941    48
CHEMBL3622821    42
CHEMBL3301607    39
CHEMBL3622820    35
CHEMBL2105759    27
CHEMBL2103743    27
CHEMBL5993958    27
CHEMBL3911726    20
CHEMBL3915630    20
CHEMBL5969212    20
CHEMBL5095079    20
CHEMBL6056544    19
CHEMBL6019568    18
CHEMBL5779930    18
CHEMBL1078178    17
CHEMBL6012988    17
CHEMBL5973885    17
CHEMBL4435170    16
CHEMBL5841594    16
Name: count, dtype: int64

In [ ]:
from immunodash import molecules

In [26]:
mols = molecules.get_unique_molecules(final_df)

In [27]:
from immunodash import descriptors

In [29]:
mols.head()

,molecule_chembl_id,canonical_smiles
0,CHEMBL319556,O=C1NCc2c(-c3ccc(F)cc3F)cc(C3CCNCC3)cc2N1c1c(C...
1,CHEMBL538798,CC(C)N(CCC(=O)c1ccc2ccccc2c1)Cc1ccccc1.Cl
2,CHEMBL154580,C=CC(=O)c1ccc2ccccc2c1
3,CHEMBL535,CCN(CC)CCNC(=O)c1c(C)[nH]c(/C=C2\C(=O)Nc3ccc(F...
4,CHEMBL386637,CN(C)[C@H]1CCCN(c2ccc(C(F)(F)F)cc2NC(=O)c2cc(C...


In [30]:
test = descriptors.calculate_rdkit_descriptors(mols.head())

test

,molecule_chembl_id,canonical_smiles,molecular_weight,logp,tpsa,hba,hbd,rotatable_bonds,ring_count,aromatic_ring_count
0,CHEMBL319556,O=C1NCc2c(-c3ccc(F)cc3F)cc(C3CCNCC3)cc2N1c1c(C...,488.365,6.76690,44.37,2,2,3,5,3
1,CHEMBL538798,CC(C)N(CCC(=O)c1ccc2ccccc2c1)Cc1ccccc1.Cl,367.920,5.74500,20.31,2,0,7,3,3
2,CHEMBL154580,C=CC(=O)c1ccc2ccccc2c1,182.222,3.20850,17.07,1,0,2,2,2
3,CHEMBL535,CCN(CC)CCNC(=O)c1c(C)[nH]c(/C=C2\C(=O)Nc3ccc(F...,398.482,3.33494,77.23,3,3,7,3,2
4,CHEMBL386637,CN(C)[C@H]1CCCN(c2ccc(C(F)(F)F)cc2NC(=O)c2cc(C...,526.538,4.39920,87.38,6,2,4,4,3


In [31]:
test.shape

(5, 10)

In [32]:
test.dtypes

molecule_chembl_id         str
canonical_smiles           str
molecular_weight       float64
logp                   float64
tpsa                   float64
hba                      int64
hbd                      int64
rotatable_bonds          int64
ring_count               int64
aromatic_ring_count      int64
dtype: object

In [33]:
test.isna().sum()

molecule_chembl_id     0
canonical_smiles       0
molecular_weight       0
logp                   0
tpsa                   0
hba                    0
hbd                    0
rotatable_bonds        0
ring_count             0
aromatic_ring_count    0
dtype: int64

In [34]:
descriptor_df = descriptors.calculate_rdkit_descriptors(mols)

In [35]:
descriptor_df.shape

(12929, 10)